# Apr 9th — presentation EDA (cache-only)

**No Ollama, no embeddings.** This notebook reads only:

- Event panel: `FINAL.csv` (numeric/metadata columns only — not full transcript text)
- Cached **full_turns** novelty records in `pkl_cache/` (e.g. `novelty_scores_full_turns_novelty_all_mini.pkl`)
- Cached LLM intent: `pkl_cache/llm_intent_cache_project_spec_turns.pkl`

Salience is recomputed locally (can take **~15–25 minutes** on the full high-novelty slice); macro-adjusted `adjusted_novelty` then matches the main pipeline definition.

Run with working directory **`final_project/`** (same as the main notebook).

In [1]:
# --- Config ---
from pathlib import Path

NOTEBOOK_ROOT = Path(".").resolve()
PKL = NOTEBOOK_ROOT / "pkl_cache"

# Full event panel (avoid reading full_transcript_text — loaded columns only)
FINAL_CSV = NOTEBOOK_ROOT / "FINAL.csv"
FINAL_USECOLS = [
    "transcriptid",
    "ticker",
    "fiscal_year",
    "fiscal_quarter",
    "call_date",
    "close_to_open_return",
]

# Full-universe full_turns novelty (on-disk cache from the main notebook)
NOVELTY_PICKLE = PKL / "novelty_scores_full_turns_novelty_all_mini.pkl"

# LLM YES/NO cache (append another path if you keep a supplemental pickle)
LLM_CACHE_FILES = [PKL / "llm_intent_cache_project_spec_turns.pkl"]

NOVELTY_THRESHOLD = 0.20
SALIENCE_THRESHOLD = 0.30
MACRO_PENALTY_F0 = 0.40
MACRO_PENALTY_K = 15.0

TOP_OVERNIGHT_CALLS = 15
TOP_PASSAGES_PER_CALL = 5

for p in (FINAL_CSV, NOVELTY_PICKLE):
    assert p.is_file(), f"Missing required file: {p}"

print("ROOT", NOTEBOOK_ROOT)
print("FINAL", FINAL_CSV.name)
print("NOVELTY", NOVELTY_PICKLE.name)

ROOT /Users/zacgarland/r_projects/wrds/final_project
FINAL FINAL.csv
NOVELTY novelty_scores_full_turns_novelty_all_mini.pkl


In [2]:
import json
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import novelty_helpers as nh
from salience_dictionary import score_sentence

try:
    from IPython.display import display
except ImportError:
    display = print

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 14

In [3]:
# --- Load FINAL (lean columns) ---
final_df = pd.read_csv(FINAL_CSV, usecols=FINAL_USECOLS, low_memory=False)
final_df["transcriptid"] = pd.to_numeric(final_df["transcriptid"], errors="coerce")
for c in ("close_to_open_return",):
    if c in final_df.columns:
        final_df[c] = pd.to_numeric(final_df[c], errors="coerce")

# Dev FINAL extract uses fiscal_year/fiscal_quarter; novelty tables use quarter_str
if "quarter_str" not in final_df.columns and {"fiscal_year", "fiscal_quarter"} <= set(final_df.columns):
    _fy = pd.to_numeric(final_df["fiscal_year"], errors="coerce")
    _fq = pd.to_numeric(final_df["fiscal_quarter"], errors="coerce")
    final_df["quarter_str"] = (
        _fy.fillna(0).astype(int).astype(str) + "Q" + _fq.fillna(0).astype(int).astype(str)
    )

print(final_df.shape)
print("n_tickers", final_df["ticker"].nunique())

(181305, 7)
n_tickers 6716


In [4]:
# --- Load novelty records + LLM caches ---
with open(NOVELTY_PICKLE, "rb") as f:
    novelty_recs = pickle.load(f)
novelty_df = pd.DataFrame(novelty_recs)
novelty_df["transcriptid"] = pd.to_numeric(novelty_df["transcriptid"], errors="coerce")

intent_cache: dict = {}
for lp in LLM_CACHE_FILES:
    if not lp.is_file():
        print("skip missing", lp.name)
        continue
    with open(lp, "rb") as f:
        chunk = pickle.load(f)
    intent_cache.update(chunk)
    print(f"loaded {len(chunk):,} keys from {lp.name}")
print(f"merged LLM keys: {len(intent_cache):,}")

print(novelty_df.shape, novelty_df["novelty_mode"].value_counts().head())

loaded 72,672 keys from llm_intent_cache_project_spec_turns.pkl
merged LLM keys: 72,672
(147026, 10) novelty_mode
full_turns    147026
Name: count, dtype: int64


In [5]:
# --- Salience + thresholds (local; not cached on disk) ---
high = novelty_df[novelty_df["novelty_score"] >= NOVELTY_THRESHOLD].copy()
print(f">= novelty {NOVELTY_THRESHOLD}: {len(high):,} turn segments")

rows = []
n = len(high)
for i, (_, row) in enumerate(high.iterrows(), start=1):
    sent = row["sentence"]
    yr = int(row["fiscal_year"]) if pd.notna(row.get("fiscal_year")) else 2020
    score, breakdown = score_sentence(str(sent), year=yr)
    top_cluster = max(breakdown, key=breakdown.get) if breakdown else None
    rows.append(
        {
            "salience_score": float(score),
            "top_cluster": top_cluster,
            "cluster_breakdown": json.dumps(breakdown),
        }
    )
    if i % 500 == 0 or i == n:
        print(f"  salience {i:,} / {n:,}")

salience_df = pd.DataFrame(rows)
cand = pd.concat([high.reset_index(drop=True), salience_df], axis=1)
cand = cand[cand["salience_score"] >= SALIENCE_THRESHOLD].copy()
print(f">= salience {SALIENCE_THRESHOLD}: {len(cand):,}")

cand["llm_cached"] = cand["sentence"].map(lambda s: s in intent_cache)
cand["llm_yes"] = cand["sentence"].map(lambda s: bool(intent_cache[s]) if s in intent_cache else False)
flagged = cand[cand["llm_yes"]].copy()
print(f"LLM YES (from merged cache): {len(flagged):,} | cached NO: {(cand['llm_cached'] & ~cand['llm_yes']).sum():,} | not in cache: {(~cand['llm_cached']).sum():,}")

>= novelty 0.2: 147,010 turn segments
  salience 500 / 147,010
  salience 1,000 / 147,010
  salience 1,500 / 147,010
  salience 2,000 / 147,010
  salience 2,500 / 147,010
  salience 3,000 / 147,010
  salience 3,500 / 147,010
  salience 4,000 / 147,010
  salience 4,500 / 147,010
  salience 5,000 / 147,010
  salience 5,500 / 147,010
  salience 6,000 / 147,010
  salience 6,500 / 147,010
  salience 7,000 / 147,010
  salience 7,500 / 147,010
  salience 8,000 / 147,010
  salience 8,500 / 147,010
  salience 9,000 / 147,010
  salience 9,500 / 147,010
  salience 10,000 / 147,010
  salience 10,500 / 147,010
  salience 11,000 / 147,010
  salience 11,500 / 147,010
  salience 12,000 / 147,010
  salience 12,500 / 147,010
  salience 13,000 / 147,010
  salience 13,500 / 147,010
  salience 14,000 / 147,010
  salience 14,500 / 147,010
  salience 15,000 / 147,010
  salience 15,500 / 147,010
  salience 16,000 / 147,010
  salience 16,500 / 147,010
  salience 17,000 / 147,010
  salience 17,500 / 147,010
  s

KeyboardInterrupt: 

In [ ]:
# --- Macro-adjusted novelty (same as main notebook) ---
signal_df, cross_section = nh.macro_adjust_signal_pipeline(
    flagged,
    penalty_f0=MACRO_PENALTY_F0,
    penalty_k=MACRO_PENALTY_K,
)
print(signal_df.shape)
display_cols = ["adjusted_novelty", "novelty_score", "salience_score", "penalty_mult", "ticker", "quarter_str", "transcriptid"]
print(signal_df[display_cols].describe().round(4))
cross_section.head(8)

## EDA — distributions and coverage

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

ax = axes[0, 0]
ax.hist(novelty_df["novelty_score"].clip(0, 1), bins=40, color="steelblue", edgecolor="white")
ax.set_title("Novelty score (all cached turns)")
ax.set_xlabel("novelty_score")

ax = axes[0, 1]
novelty_df["ticker"].value_counts().sort_index().plot(kind="bar", ax=ax, color="coral")
ax.set_title("Turn-segments per ticker (all novelty rows)")

ax = axes[1, 0]
r = final_df["close_to_open_return"].dropna()
ax.hist(r.clip(-0.15, 0.15), bins=40, color="seagreen", edgecolor="white")
ax.set_title("Overnight return (clipped ±15% for viz)")
ax.set_xlabel("close_to_open_return")

ax = axes[1, 1]
ax.scatter(cand["novelty_score"], cand["salience_score"], alpha=0.25, s=12, c="purple")
ax.set_xlabel("novelty_score")
ax.set_ylabel("salience_score")
ax.set_title("Novelty vs salience (post novelty cutoff)")

plt.tight_layout()
plt.show()

# Pipeline funnel
funnel = pd.DataFrame(
    [
        ("all novelty rows", len(novelty_df)),
        (f"novelty ≥ {NOVELTY_THRESHOLD}", len(high)),
        (f"+ salience ≥ {SALIENCE_THRESHOLD}", len(cand)),
        ("+ LLM YES (cached)", len(flagged)),
        ("segment-level after macro merge", len(signal_df)),
    ],
    columns=["stage", "n"],
)
display(funnel)

In [ ]:
# Call-level: max novelty vs overnight (merge authoritative return from FINAL)
call_ret = final_df[["transcriptid", "ticker", "quarter_str", "close_to_open_return"]].drop_duplicates("transcriptid")
per_call = (
    novelty_df.groupby("transcriptid", as_index=False)
    .agg(max_novelty=("novelty_score", "max"), n_turns=("sentence", "count"))
    .merge(call_ret, on="transcriptid", how="inner")
)
per_call = per_call.dropna(subset=["close_to_open_return"])

fig, ax = plt.subplots(figsize=(9, 5))
sc = ax.scatter(
    per_call["max_novelty"],
    per_call["close_to_open_return"] * 100,
    c=np.abs(per_call["close_to_open_return"]),
    cmap="viridis",
    alpha=0.6,
    s=28,
)
plt.colorbar(sc, ax=ax, label="|overnight| (decimal)")
ax.axhline(0, color="gray", lw=0.8)
ax.set_xlabel("Max novelty_score per call")
ax.set_ylabel("Overnight return (%)")
ax.set_title("Call-level: max full-turn novelty vs overnight return")
plt.tight_layout()
plt.show()

## Largest |overnight| calls → top adjusted novelty & flagged passages

In [ ]:
fd = final_df.dropna(subset=["close_to_open_return", "transcriptid"]).copy()
fd["abs_overnight"] = fd["close_to_open_return"].abs()
top_calls = fd.nlargest(TOP_OVERNIGHT_CALLS, "abs_overnight")[
    ["ticker", "quarter_str", "transcriptid", "close_to_open_return", "abs_overnight", "call_date"]
].reset_index(drop=True)
top_calls

In [ ]:
# For each spotlight call: top passages among LLM-YES segments by adjusted_novelty
spotlight_rows = []
for rank, cr in top_calls.iterrows():
    tid = int(cr["transcriptid"])
    sub = signal_df[signal_df["transcriptid"] == tid].sort_values("adjusted_novelty", ascending=False)
    if sub.empty:
        spotlight_rows.append(
            {
                "overnight_rank": rank + 1,
                "ticker": cr["ticker"],
                "quarter_str": cr["quarter_str"],
                "transcriptid": tid,
                "close_to_open_return": cr["close_to_open_return"],
                "n_llm_yes_segments": 0,
                "top_adjusted_novelty": np.nan,
                "passage_preview": "(no LLM-YES cached segments for this call after novelty/salience gates)",
            }
        )
        continue
    top = sub.head(TOP_PASSAGES_PER_CALL)
    for j, (_, seg) in enumerate(top.iterrows(), start=1):
        txt = str(seg["sentence"])
        spotlight_rows.append(
            {
                "overnight_rank": rank + 1,
                "passage_rank": j,
                "ticker": cr["ticker"],
                "quarter_str": cr["quarter_str"],
                "transcriptid": tid,
                "close_to_open_return": cr["close_to_open_return"],
                "adjusted_novelty": seg["adjusted_novelty"],
                "novelty_score": seg["novelty_score"],
                "salience_score": seg["salience_score"],
                "penalty_mult": seg["penalty_mult"],
                "top_cluster": seg["top_cluster"],
                "chunk_index": seg.get("chunk_index"),
                "passage": txt[:1200] + ("…" if len(txt) > 1200 else ""),
            }
        )

spot_tbl = pd.DataFrame(spotlight_rows)
pd.set_option("display.max_colwidth", 120)
spot_tbl

In [ ]:
# Bar chart: spotlight overnight moves (%)
plot_df = top_calls.copy()
plot_df["overnight_pct"] = plot_df["close_to_open_return"] * 100
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(
    plot_df.index,
    plot_df["overnight_pct"],
    color=np.where(plot_df["overnight_pct"] >= 0, "tab:green", "tab:red"),
)
ax.set_xticks(plot_df.index)
ax.set_xticklabels(
    [f"{r.ticker}\n{r.quarter_str}" for _, r in plot_df.iterrows()],
    rotation=45,
    ha="right",
    fontsize=9,
)
ax.axhline(0, color="black", lw=0.6)
ax.set_ylabel("Overnight return (%)")
ax.set_title(f"Top {TOP_OVERNIGHT_CALLS} calls by |close_to_open_return|")
plt.tight_layout()
plt.show()

### Notes

- **LLM YES** uses only strings present in the merged pickle; segments never scored by the LLM are omitted from `signal_df` here.
- **Novelty pickle** must align with **full_turns** text; the filename suffix (`nomic` vs `all_mini`) reflects the embedding run used to build that cache.
- Smaller **dev** pickles (`FINAL__dev_*.csv`, `novelty_scores_*__dev_*.pkl`) are optional: point `FINAL_CSV` / `FINAL_USECOLS` / `NOVELTY_PICKLE` / `LLM_CACHE_FILES` at those paths for a fast dry run.